In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [2]:
def evaluate_models(X_train, X_test, y_train, y_test):

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "KNN": KNeighborsClassifier(),
        "SVM": SVC(),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "XGBoost": XGBClassifier(
            eval_metric='logloss',
            random_state=42
        )
    }

    results = []
    for name, model in models.items():
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')
        cm = confusion_matrix(y_test, y_pred)

        print("\n", "="*50)
        print(name)
        print("="*50)
        print("Confusion Matrix")
        print(cm)

        results.append([name, acc, precision, recall, f1])

    summary = pd.DataFrame(
        results,
        columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score"]
    )

    return summary.sort_values(by="Accuracy",ascending=False)

# 1.

In [3]:
df = pd.read_csv("/home/sav/Desktop/Machine_Learning/Assignments/Datasets/iris.csv")
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [4]:
X = df.drop("species", axis=1)
y = df["species"]
le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

summary = evaluate_models(
    X_train,
    X_test,
    y_train,
    y_test
)
print(summary)


Logistic Regression
Confusion Matrix
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

KNN
Confusion Matrix
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

SVM
Confusion Matrix
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

Decision Tree
Confusion Matrix
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

Random Forest
Confusion Matrix
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

AdaBoost
Confusion Matrix
[[10  0  0]
 [ 0  8  1]
 [ 0  1 10]]

XGBoost
Confusion Matrix
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  1.000000   1.000000  1.000000  1.000000
1                  KNN  1.000000   1.000000  1.000000  1.000000
2                  SVM  1.000000   1.000000  1.000000  1.000000
3        Decision Tree  1.000000   1.000000  1.000000  1.000000
4        Random Forest  1.000000   1.000000  1.000000  1.000000
6              XGBoost  1.000000   1.000000  1.000000  1.000000
5             AdaBoost  0.933333   0.933333  0.933333  0.933333


In [5]:
best_model = summary.iloc[0]
print(best_model)

Model        Logistic Regression
Accuracy                     1.0
Precision                    1.0
Recall                       1.0
F1 Score                     1.0
Name: 0, dtype: object


# 2.

In [6]:
df = pd.read_csv("/home/sav/Desktop/Machine_Learning/Assignments/Datasets/titanic.csv")
df.drop(["PassengerId","Name","Ticket","Cabin"],axis=1,inplace=True)
X = df.drop("Survived", axis=1)
y = df["Survived"]

cat_cols = X.select_dtypes(
    include="object"
).columns

num_cols = X.select_dtypes(
    exclude="object"
).columns

preprocessor = ColumnTransformer([

    (
        "num",
        Pipeline([
            ("imputer",
             SimpleImputer(strategy="median")),
            ("scaler",
             StandardScaler())
        ]),
        num_cols
    ),

    (
        "cat",
        Pipeline([
            ("imputer",
             SimpleImputer(strategy="most_frequent")),
            ("encoder",
             OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols
    )
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

summary = evaluate_models(
    X_train,
    X_test,
    y_train,
    y_test
)
print(summary)


Logistic Regression
Confusion Matrix
[[90 15]
 [19 55]]

KNN
Confusion Matrix
[[91 14]
 [20 54]]

SVM
Confusion Matrix
[[92 13]
 [20 54]]

Decision Tree
Confusion Matrix
[[83 22]
 [18 56]]

Random Forest
Confusion Matrix
[[89 16]
 [19 55]]

AdaBoost
Confusion Matrix
[[91 14]
 [22 52]]

XGBoost
Confusion Matrix
[[87 18]
 [19 55]]
                 Model  Accuracy  Precision    Recall  F1 Score
2                  SVM  0.815642   0.815038  0.815642  0.814040
0  Logistic Regression  0.810056   0.809163  0.810056  0.809193
1                  KNN  0.810056   0.809194  0.810056  0.808681
4        Random Forest  0.804469   0.803641  0.804469  0.803824
5             AdaBoost  0.798883   0.798104  0.798883  0.796827
6              XGBoost  0.793296   0.792920  0.793296  0.793083
3        Decision Tree  0.776536   0.778857  0.776536  0.777307


In [7]:
best_model = summary.iloc[0]
print(best_model)

Model             SVM
Accuracy     0.815642
Precision    0.815038
Recall       0.815642
F1 Score      0.81404
Name: 2, dtype: object


# 3.

In [8]:
df = pd.read_csv("/home/sav/Desktop/Machine_Learning/Assignments/Datasets/diabetes.csv")
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

summary = evaluate_models(
    X_train,
    X_test,
    y_train,
    y_test
)
print(summary)


Logistic Regression
Confusion Matrix
[[79 20]
 [18 37]]

KNN
Confusion Matrix
[[79 20]
 [27 28]]

SVM
Confusion Matrix
[[82 17]
 [24 31]]

Decision Tree
Confusion Matrix
[[75 24]
 [15 40]]

Random Forest
Confusion Matrix
[[77 22]
 [21 34]]

AdaBoost
Confusion Matrix
[[80 19]
 [15 40]]

XGBoost
Confusion Matrix
[[72 27]
 [16 39]]
                 Model  Accuracy  Precision    Recall  F1 Score
5             AdaBoost  0.779221   0.783484  0.779221  0.780818
0  Logistic Regression  0.753247   0.755394  0.753247  0.754191
3        Decision Tree  0.746753   0.758929  0.746753  0.750300
2                  SVM  0.733766   0.727959  0.733766  0.729265
4        Random Forest  0.720779   0.721939  0.720779  0.721328
6              XGBoost  0.720779   0.737013  0.720779  0.725259
1                  KNN  0.694805   0.687444  0.694805  0.689645


In [9]:
best_model = summary.iloc[0]
print(best_model)

Model        AdaBoost
Accuracy     0.779221
Precision    0.783484
Recall       0.779221
F1 Score     0.780818
Name: 5, dtype: object


# 4.

In [10]:
df = pd.read_csv("/home/sav/Desktop/Machine_Learning/Assignments/Datasets/ObesityLevel.csv")
X = df.drop("NObeyesdad", axis=1)
y = df["NObeyesdad"]
le = LabelEncoder()
y = le.fit_transform(y)

cat_cols = X.select_dtypes(
    include="object"
).columns

num_cols = X.select_dtypes(
    exclude="object"
).columns

preprocessor = ColumnTransformer([

    (
        "num",
        Pipeline([
            ("imputer",
             SimpleImputer(strategy="median")),
            ("scaler",
             StandardScaler())
        ]),
        num_cols
    ),

    (
        "cat",
        Pipeline([
            ("imputer",
             SimpleImputer(strategy="most_frequent")),
            ("encoder",
             OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols
    )
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

summary = evaluate_models(
    X_train,
    X_test,
    y_train,
    y_test
)
print(summary)


Logistic Regression
Confusion Matrix
[[56  0  0  0  0  0  0]
 [11 39  0  0  0  9  3]
 [ 0  0 70  6  0  0  2]
 [ 0  0  2 56  0  0  0]
 [ 0  0  0  0 63  0  0]
 [ 0  4  0  0  0 42 10]
 [ 0  0  3  0  0  5 42]]

KNN
Confusion Matrix
[[53  2  0  0  0  1  0]
 [15 19  8  2  0 10  8]
 [ 0  0 74  2  0  0  2]
 [ 0  0  2 56  0  0  0]
 [ 0  0  0  0 63  0  0]
 [ 2  5  0  0  0 46  3]
 [ 0  1  6  3  1  3 36]]

SVM
Confusion Matrix
[[54  2  0  0  0  0  0]
 [ 2 54  1  0  0  4  1]
 [ 0  0 75  3  0  0  0]
 [ 0  0  1 57  0  0  0]
 [ 0  0  0  0 63  0  0]
 [ 0  8  0  0  0 47  1]
 [ 0  0  1  0  0  5 44]]

Decision Tree
Confusion Matrix
[[54  2  0  0  0  0  0]
 [ 5 55  0  0  0  2  0]
 [ 0  0 75  2  0  0  1]
 [ 0  0  3 55  0  0  0]
 [ 0  0  0  0 63  0  0]
 [ 0  6  0  0  0 50  0]
 [ 0  0  1  0  0  3 46]]

Random Forest
Confusion Matrix
[[54  2  0  0  0  0  0]
 [ 1 56  0  0  0  4  1]
 [ 0  2 73  1  0  1  1]
 [ 0  0  1 57  0  0  0]
 [ 0  0  0  0 63  0  0]
 [ 0  6  0  0  0 49  1]
 [ 0  1  0  0  0  2 47]]

AdaBoost

/home/sav/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



XGBoost
Confusion Matrix
[[56  0  0  0  0  0  0]
 [ 5 50  0  0  0  7  0]
 [ 0  0 75  2  0  1  0]
 [ 0  0  2 56  0  0  0]
 [ 0  0  0  0 63  0  0]
 [ 0  3  0  0  0 53  0]
 [ 0  0  0  0  0  2 48]]
                 Model  Accuracy  Precision    Recall  F1 Score
6              XGBoost  0.947991   0.950320  0.947991  0.947673
4        Random Forest  0.943262   0.945032  0.943262  0.943823
3        Decision Tree  0.940898   0.941471  0.940898  0.940957
2                  SVM  0.931442   0.932006  0.931442  0.931466
0  Logistic Regression  0.869976   0.874865  0.869976  0.867157
1                  KNN  0.820331   0.811825  0.820331  0.802044
5             AdaBoost  0.508274   0.471467  0.508274  0.457148


In [11]:
best_model = summary.iloc[0]
print(best_model)

Model         XGBoost
Accuracy     0.947991
Precision     0.95032
Recall       0.947991
F1 Score     0.947673
Name: 6, dtype: object


# 5.

In [12]:
df = pd.read_csv("/home/sav/Desktop/Machine_Learning/Assignments/Datasets/heart.csv")
X = df.drop("target", axis=1)
y = df["target"]

cat_cols = X.select_dtypes(
    include="object"
).columns

num_cols = X.select_dtypes(
    exclude="object"
).columns

preprocessor = ColumnTransformer([

    (
        "num",
        Pipeline([
            ("imputer",
             SimpleImputer(strategy="median")),
            ("scaler",
             StandardScaler())
        ]),
        num_cols
    ),

    (
        "cat",
        Pipeline([
            ("imputer",
             SimpleImputer(strategy="most_frequent")),
            ("encoder",
             OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols
    )
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

summary = evaluate_models(
    X_train,
    X_test,
    y_train,
    y_test
)
print(summary)


Logistic Regression
Confusion Matrix
[[25  4]
 [ 5 27]]

KNN
Confusion Matrix
[[27  2]
 [ 4 28]]

SVM
Confusion Matrix
[[26  3]
 [ 5 27]]

Decision Tree
Confusion Matrix
[[25  4]
 [11 21]]

Random Forest
Confusion Matrix
[[24  5]
 [ 5 27]]

AdaBoost
Confusion Matrix
[[26  3]
 [ 8 24]]

XGBoost
Confusion Matrix
[[25  4]
 [ 7 25]]
                 Model  Accuracy  Precision    Recall  F1 Score
1                  KNN  0.901639   0.903684  0.901639  0.901692
2                  SVM  0.868852   0.870862  0.868852  0.868923
0  Logistic Regression  0.852459   0.853076  0.852459  0.852538
4        Random Forest  0.836066   0.836066  0.836066  0.836066
5             AdaBoost  0.819672   0.829851  0.819672  0.819187
6              XGBoost  0.819672   0.823647  0.819672  0.819672
3        Decision Tree  0.754098   0.770801  0.754098  0.752240


In [13]:
best_model = summary.iloc[0]
print(best_model)

Model             KNN
Accuracy     0.901639
Precision    0.903684
Recall       0.901639
F1 Score     0.901692
Name: 1, dtype: object
